# 01 · Conducta de moondream2: respuestas, oclusión y grounding

Este notebook **solo hace preguntas** al modelo; no mira nada interno. Usa el conjunto `conjunto_v2.zip` (120 preguntas × 4 versiones de imagen = 480 imágenes).

Para cada pregunta:
- **query** en las 4 versiones: `c0` (original), `c1` (sin 1 figura objetivo), `c2` (sin 2), `control` (sin 1 figura que no es objetivo);
- **point** en `c0`: el modelo marca dónde ve las figuras objetivo.

Las reglas del experimento están en `DISENO.md`.

**Cómo correrlo**
1. En Google Drive deben existir `memoria_explicabilidad/datos/conjunto_v2.zip` y este notebook.
2. *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*.
3. *Entorno de ejecución → Ejecutar todas*. Acepta el permiso de Drive.
4. Si la sesión se reinicia sola (pasa una vez, al instalar `transformers`), vuelve a *Ejecutar todas*.
5. Al final aparece **LISTO**. Los resultados quedan en `memoria_explicabilidad/salidas/01_conducta/`.

Si se corta a mitad de camino, vuelve a *Ejecutar todas*: retoma desde donde quedó.

In [ ]:
# Fija la versión de transformers usada en todo el proyecto. Si hay que instalarla, la sesión se reinicia sola.
import importlib.metadata, os
VERSION = "4.46.3"
if importlib.metadata.version("transformers") != VERSION:
    os.system(f"pip install -q transformers=={VERSION}")
    print("Instalado. La sesión se reinicia: vuelve a 'Ejecutar todas'.")
    os.kill(os.getpid(), 9)
print("transformers", VERSION, "OK")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/memoria_explicabilidad"
ZIP_CONJUNTO = f"{BASE}/datos/conjunto_v2.zip"
SALIDA = f"{BASE}/salidas/01_conducta"
os.makedirs(SALIDA, exist_ok=True)

import zipfile
with zipfile.ZipFile(ZIP_CONJUNTO) as z:
    z.extractall("/content")
CONJUNTO = "/content/conjunto_v2"
print(len(os.listdir(f"{CONJUNTO}/imagenes")), "imágenes")

In [ ]:
import torch
from transformers import AutoModelForCausalLM

MODEL_ID = "vikhyatk/moondream2"
MODEL_REVISION = "2025-06-21"
assert torch.cuda.is_available(), "Sin GPU: Entorno de ejecución → Cambiar tipo de entorno → T4 GPU"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, trust_remote_code=True,
    torch_dtype=torch.bfloat16, device_map={"": "cuda"})
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import re

SUFIJO = " Respond with only the integer number."
AJUSTES_QUERY = {"temperature": 0, "max_tokens": 16}   # temperature 0 = siempre la misma respuesta
AJUSTES_POINT = {"max_objects": 100}

def extraer_entero(texto):
    """Primer entero, ignorando marcadores de lista ('1.', '2)') al inicio de línea. None si no hay."""
    limpio = re.sub(r"(?m)^\s*\d+[.)]\s+", "", str(texto))
    m = re.search(r"-?\d+", limpio)
    return int(m.group()) if m else None

def preguntar(enc, pregunta):
    # se pasa la imagen ya codificada: query(PIL, settings=...) falla porque exige la clave "variant"
    return model.query(enc, pregunta + SUFIJO, settings=AJUSTES_QUERY)["answer"]

def apuntar(enc, objeto):
    return model.point(enc, objeto, settings=AJUSTES_POINT)["points"]

In [ ]:
import csv, json, time
from PIL import Image

preguntas = {p["qid"]: p for p in csv.DictReader(open(f"{CONJUNTO}/preguntas.csv"))}
imagenes = list(csv.DictReader(open(f"{CONJUNTO}/imagenes.csv")))

RUTA_RESP = f"{SALIDA}/respuestas.csv"
RUTA_PUNTOS = f"{SALIDA}/puntos.csv"
CAMPOS_RESP = ["qid", "condicion", "archivo", "gt", "respuesta_cruda", "prediccion", "segundos"]
CAMPOS_PUNTOS = ["qid", "n_puntos", "puntos_json"]

def ya_hechos(ruta, claves):
    if not os.path.exists(ruta):
        return set()
    return {tuple(f[k] for k in claves) for f in csv.DictReader(open(ruta))}

hechos_resp = ya_hechos(RUTA_RESP, ["qid", "condicion"])
hechos_puntos = ya_hechos(RUTA_PUNTOS, ["qid"])
print(f"Ya hechos: {len(hechos_resp)}/480 respuestas, {len(hechos_puntos)}/120 grounding")

def agregar(ruta, campos, fila):
    nuevo = not os.path.exists(ruta)
    with open(ruta, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=campos)
        if nuevo:
            w.writeheader()
        w.writerow(fila)

inicio = time.time()
for k, fila in enumerate(imagenes):
    qid, cond = fila["qid"], fila["condicion"]
    p = preguntas[qid]
    falta_resp = (qid, cond) not in hechos_resp
    falta_point = cond == "c0" and (qid,) not in hechos_puntos
    if not (falta_resp or falta_point):
        continue
    img = Image.open(f"{CONJUNTO}/imagenes/{fila['archivo']}").convert("RGB")
    enc = model.encode_image(img)
    if falta_resp:
        t0 = time.time()
        cruda = preguntar(enc, p["pregunta"])
        agregar(RUTA_RESP, CAMPOS_RESP, dict(qid=qid, condicion=cond, archivo=fila["archivo"], gt=fila["gt"],
                respuesta_cruda=cruda, prediccion=extraer_entero(cruda), segundos=round(time.time() - t0, 3)))
    if falta_point:
        puntos = apuntar(enc, p["objeto_point"])
        agregar(RUTA_PUNTOS, CAMPOS_PUNTOS, dict(qid=qid, n_puntos=len(puntos), puntos_json=json.dumps(puntos)))
    if k % 20 == 0:
        print(f"[{k+1}/480] {qid} {cond}  ({(time.time()-inicio)/60:.1f} min)")
print("Inferencia terminada.")

In [ ]:
# Control de determinismo: con temperature 0, repetir la pregunta debe dar exactamente lo mismo.
import pandas as pd
resp = pd.read_csv(RUTA_RESP)
muestra = resp[resp.condicion == "c0"].head(10)
iguales = 0
for _, f in muestra.iterrows():
    enc = model.encode_image(Image.open(f"{CONJUNTO}/imagenes/{f.archivo}").convert("RGB"))
    iguales += preguntar(enc, preguntas[f.qid]["pregunta"]) == f.respuesta_cruda
print(f"Determinismo: {iguales}/10 respuestas idénticas al repetir")

import transformers, platform
info = dict(modelo=MODEL_ID, revision=MODEL_REVISION, transformers=transformers.__version__,
            torch=torch.__version__, gpu=torch.cuda.get_device_name(0), python=platform.python_version(),
            sufijo=SUFIJO, ajustes_query=AJUSTES_QUERY, ajustes_point=AJUSTES_POINT,
            determinismo_10=int(iguales), vram_pico_mb=round(torch.cuda.max_memory_allocated() / 1024**2))
json.dump(info, open(f"{SALIDA}/info_corrida.json", "w"), indent=2)

In [ ]:
# Resumen rápido (el análisis completo se hace después, fuera de Colab)
r = resp.copy()
c0 = r[r.condicion == "c0"].set_index("qid")
print("Respuestas leídas:", r.prediccion.notna().sum(), "/", len(r))
print("Acierto en c0:", round((c0.prediccion == c0.gt).mean() * 100, 1), "%")
piv = r.pivot(index="qid", columns="condicion", values="prediccion")
for cond, esperado in [("c1", -1), ("c2", -2), ("control", 0)]:
    delta = piv[cond] - piv["c0"]
    print(f"{cond}: cambio esperado {esperado:+d} | ocurre en {round((delta == esperado).mean() * 100, 1)} % de las preguntas")
print("\nLISTO. Archivos en", SALIDA, ":", os.listdir(SALIDA))